# LeetCode 754 — Reach a Number bằng Python

Link đề: https://leetcode.com/problems/reach-a-number/ · Độ khó: **Medium** · Chủ đề: Math, Binary Search (quan sát số học: số tam giác và tính chẵn/lẻ)

Skeleton LeetCode:

```python
class Solution:
    def reachNumber(self, target: int) -> int:
        ...
```

Ghi chú: `target` lên tới `10^9` nên không thể thử các dãy trái/phải. Bài này là bài "nhìn ra quy luật số học": tổng `1 + 2 + ... + k`, việc **đổi hướng một bước** chỉ làm tổng giảm đi một số **chẵn**, và mọi số từ `0` đến tổng đều là tổng của một tập con của `{1..k}`. Notebook đi từ vét cạn dấu, BFS theo tập vị trí, tới vòng lặp đơn giản `O(√target)`, tìm kiếm nhị phân và công thức đóng.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đứng ở `0` trên trục số vô hạn; ở lần di chuyển thứ `i` (`i = 1, 2, ...`) bạn đi đúng `i` bước sang **trái hoặc phải**; tìm số lần di chuyển **ít nhất** để tới `target`.
2. Nắm các công cụ Python/toán cần dùng: giá trị tuyệt đối `abs` và tính đối xứng, **số tam giác** `k(k+1)/2`, tính chẵn lẻ (`% 2`), `itertools.product` để liệt kê dấu `±`, `math.isqrt`, tìm kiếm nhị phân trên hàm đơn điệu.
3. Đi từ **vét cạn mọi dãy dấu** → **BFS theo tập vị trí đạt được** → **tăng `k` đến khi tổng đủ lớn và cùng tính chẵn lẻ** → **nhị phân** → **công thức đóng bằng `isqrt`**, và hiểu vì sao các cách cho cùng đáp án.
4. Hiểu mấu chốt: đi sang trái thay vì sang phải ở bước `i` làm tổng **giảm `2i`** (một số chẵn). Nên với `k` bước, các vị trí đạt được là `S_k − 2F` với `F` là tổng một tập con của `{1..k}`; và **mọi** `F` từ `0` đến `S_k` đều làm được. Vậy tới được `target` (không âm) với `k` bước khi và chỉ khi `S_k ≥ target` và `S_k − target` chẵn.
5. Tránh các lỗi kinh điển: quên lấy `abs(target)`, chỉ kiểm tra `S_k ≥ target` mà quên tính chẵn lẻ, tưởng chỉ cần thêm đúng 1 bước khi lệch chẵn lẻ (thực tế có thể cần thêm 2), dùng mô phỏng tham lam, và dùng vét cạn với `target` lớn.

## 2. Đề bài & ví dụ

Bạn đang đứng ở vị trí `0` trên **trục số vô hạn**, đích ở vị trí `target`.

Bạn thực hiện một số lần di chuyển `numMoves` sao cho:

- Mỗi lần di chuyển bạn chọn đi sang **trái** hoặc **phải**.
- Ở lần di chuyển thứ `i` (từ `i = 1` đến `i = numMoves`), bạn đi `i` bước theo hướng đã chọn.

Cho số nguyên `target`, trả về **số lần di chuyển ít nhất** (tức `numMoves` nhỏ nhất) để tới đích.

**Ví dụ 1**

- Input: `target = 2`
- Output: `3`
- Giải thích: lần 1: `0 → 1` (1 bước); lần 2: `1 → -1` (2 bước); lần 3: `-1 → 2` (3 bước).

**Ví dụ 2**

- Input: `target = 3`
- Output: `2`
- Giải thích: lần 1: `0 → 1`; lần 2: `1 → 3`.

**Ràng buộc:**

- `-10^9 <= target <= 10^9`
- `target != 0`

Chú ý: ở lần `i` **phải đi đúng `i` bước** (không được bỏ qua lần nào, không được đi ít hơn). `target` có thể âm.

In [1]:
from typing import List
from itertools import product
from math import isqrt
import random

examples = [
    (2, 3),
    (3, 2),
]
for target, expected in examples:
    print(f"target = {target} -> kỳ vọng {expected}")

target = 2 -> kỳ vọng 3
target = 3 -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Số tam giác: `1 + 2 + ... + k = k(k+1)/2`

Tổng các số nguyên liên tiếp từ `1` đến `k` là **số tam giác** `S_k = k(k+1)/2`. Ví dụ `S_4 = 1 + 2 + 3 + 4 = 10 = 4·5/2`. Nếu bạn **luôn** đi sang phải thì sau `k` lần bạn đứng ở vị trí `S_k`.

Mẹo nhớ công thức: ghép đầu với cuối: `1 + k`, `2 + (k-1)`, ... mỗi cặp bằng `k + 1`, có `k/2` cặp.

In [2]:
for k in range(1, 8):
    S = k * (k + 1) // 2
    assert S == sum(range(1, k + 1))
    print(f"S_{k} = {S}")

print("S_44721 =", 44721 * 44722 // 2, "(gần 10^9)")

S_1 = 1
S_2 = 3
S_3 = 6
S_4 = 10
S_5 = 15
S_6 = 21
S_7 = 28
S_44721 = 1000006281 (gần 10^9)


### 3.2. Giá trị tuyệt đối và tính đối xứng

Trục số đối xứng qua `0`: nếu ta tới được `+T` bằng một dãy trái/phải thì **đổi hướng mọi bước** cho ta dãy tới `-T`. Do đó số lần di chuyển ít nhất tới `target` bằng số lần ít nhất tới `-target`. Ta chỉ cần xét `T = abs(target) > 0`.

In [3]:
for t in (5, -5, 3, -3):
    print(t, "->", abs(t))

# Đổi hướng mọi bước: vị trí đổi dấu
signs = [1, -1, 1]                        # +1 -2 +3 = 2
print(sum(s * i for s, i in zip(signs, range(1, 4))))
flipped = [-s for s in signs]
print(sum(s * i for s, i in zip(flipped, range(1, 4))))   # -2

5 -> 5
-5 -> 5
3 -> 3
-3 -> 3
2
-2


### 3.3. Tính chẵn/lẻ và "đổi hướng một bước làm tổng giảm số chẵn"

Một số nguyên chẵn nếu `x % 2 == 0`, lẻ nếu `x % 2 == 1`. Tính chất cần dùng: **chẵn ± chẵn = chẵn**, **lẻ ± lẻ = chẵn**, **chẵn ± lẻ = lẻ**.

Giả sử ta đi **tất cả sang phải** được `S_k`. Nếu ta đổi hướng bước thứ `i` (sang trái), vị trí cuối thay vì `+i` thì có `-i`: giảm đi `2i`. Đổi hướng một tập các bước `F` thì vị trí cuối là `S_k − 2·sum(F)`. Vì `2·sum(F)` luôn **chẵn**, vị trí cuối luôn **cùng tính chẵn/lẻ** với `S_k`. Đây là điều kiện **cần**: tới được `T` với `k` bước thì `S_k ≡ T (mod 2)`.

In [4]:
k = 4
S = k * (k + 1) // 2                           # 10
print("S =", S, "| chẵn?", S % 2 == 0)
for signs in product((1, -1), repeat=k):
    pos = sum(s * i for s, i in zip(signs, range(1, k + 1)))
    assert (pos - S) % 2 == 0                  # mọi vị trí cùng tính chẵn lẻ với S
print("với k = 4 mọi 16 dãy dấu đều cho vị trí cùng chẵn/lẻ với S = 10")

S = 10 | chẵn? True
với k = 4 mọi 16 dãy dấu đều cho vị trí cùng chẵn/lẻ với S = 10


### 3.4. `itertools.product` để liệt kê mọi dãy dấu

`product((1, -1), repeat=k)` sinh mọi dãy `k` dấu `±1`: có `2^k` dãy. Ta dùng nó như phương pháp vét cạn ở quy mô nhỏ (`k ≤ 10`).

In [5]:
print(list(product((1, -1), repeat=2)))
print(len(list(product((1, -1), repeat=10))), "= 2**10 =", 2 ** 10)

[(1, 1), (1, -1), (-1, 1), (-1, -1)]
1024 = 2**10 = 1024


### 3.5. `math.isqrt` và giải bất phương trình `k(k+1)/2 ≥ T`

`math.isqrt(n)` trả về phần nguyên của căn bậc hai (số nguyên, chính xác, không lỗi làm tròn như `int(n ** 0.5)`). Bất phương trình `k(k+1)/2 ≥ T` tương đương `(2k+1)^2 ≥ 8T + 1`, nên `k` nhỏ nhất thoả mãn là phần nguyên trần của `(√(8T+1) − 1)/2`. Dùng `isqrt` để tính chính xác (sẽ làm ở Cách 5).

**Tìm kiếm nhị phân** thì quen thuộc: `S_k` tăng nghiêm ngặt theo `k`, nên có thể nhị phân tìm `k` nhỏ nhất với `S_k ≥ T` (làm ở Cách 4).

In [6]:
print(isqrt(17), isqrt(16), isqrt(10 ** 18))       # 4 4 1000000000
print(int((10 ** 18) ** 0.5), int((10 ** 18 + 1) ** 0.5))   # có thể lệch ở số lớn: dùng isqrt cho chắc

T = 5
r = isqrt(8 * T + 1)                              # isqrt(41) = 6
print("r =", r, "-> k nhỏ nhất với S_k >= 5 là 3 (S_3 = 6)")

4 4 1000000000
1000000000 1000000000
r = 6 -> k nhỏ nhất với S_k >= 5 là 3 (S_3 = 6)


## 4. Ý tưởng nền tảng / trực giác

Tính tay Ví dụ 1 (`target = 2`):

- `k = 1`: `S_1 = 1 < 2`: chưa với tới.
- `k = 2`: `S_2 = 3 ≥ 2`, nhưng `S_2 − 2 = 1` **lẻ**: không thể (đổi hướng làm tổng giảm số chẵn nên không thể giảm từ `3` xuống `2`).
- `k = 3`: `S_3 = 6 ≥ 2`, `S_3 − 2 = 4` chẵn. Cần đổi hướng tập các bước có tổng `4 / 2 = 2`: đổi bước `2`: `+1 − 2 + 3 = 2` ✓.

Vậy đáp án `3`. Tính tay Ví dụ 2 (`target = 3`): `k = 2`: `S_2 = 3 ≥ 3`, hiệu `0` chẵn → không cần đổi hướng bước nào → đáp án `2`.

Quy luật: **tìm `k` nhỏ nhất sao cho `S_k ≥ T` và `S_k − T` chẵn.** Hai điều kiện đều **cần** (đã giải thích) và (sẽ chứng minh ở mục 10) cũng **đủ**.

In [7]:
T = 2
for k in range(1, 6):
    S = k * (k + 1) // 2
    ok = S >= T and (S - T) % 2 == 0
    print(f"k={k}: S={S}, S>=T? {S >= T}, (S-T) chẵn? {(S - T) % 2 == 0} -> {'THỎA' if ok else 'không'}")

k=1: S=1, S>=T? False, (S-T) chẵn? False -> không
k=2: S=3, S>=T? True, (S-T) chẵn? False -> không
k=3: S=6, S>=T? True, (S-T) chẵn? True -> THỎA
k=4: S=10, S>=T? True, (S-T) chẵn? True -> THỎA
k=5: S=15, S>=T? True, (S-T) chẵn? False -> không


## 5. Cách 1 — Brute Force: thử mọi dãy dấu

### Ý tưởng

Với mỗi `k = 1, 2, 3, ...`, thử **mọi** dãy dấu `±` (có `2^k` dãy), tính vị trí cuối; nếu có dãy cho đúng `T = |target|` thì `k` là đáp án (vì ta thử `k` tăng dần, `k` đầu tiên thành công là nhỏ nhất).

### Code

In [8]:
def reach_brute(target):
    T = abs(target)
    k = 0
    while True:
        k += 1
        for signs in product((1, -1), repeat=k):
            if sum(s * i for s, i in zip(signs, range(1, k + 1))) == T:
                return k

for target, expected in examples:
    got = reach_brute(target)
    print(got, expected, "OK" if got == expected else "SAI")
print([reach_brute(t) for t in range(1, 13)])

3 3 OK
2 2 OK
[1, 3, 2, 3, 5, 3, 5, 4, 5, 4, 5, 7]


### Độ phức tạp và nút thắt

- **Time:** `O(2^k · k)` với `k` là đáp án. Vì `k ~ √(2T)`, ngay `T = 1000` đã có `k ≈ 45` và `2^45` dãy — không chạy nổi.
- **Space:** `O(k)`.

**Nút thắt:** ta liệt kê cả những dãy dẫn tới cùng vị trí. Nhiều dãy khác nhau cho cùng vị trí; chỉ cần biết **tập các vị trí đạt được** sau mỗi bước. Đó là Cách 2.

## 6. Ý tưởng thuật toán / bước đột phá

Hai quan sát nối tiếp nhau:

**Quan sát A (điều kiện cần).** Sau `k` bước, vị trí cuối là `S_k − 2F` với `F` là tổng một tập con của `{1, ..., k}` (các bước đi sang trái). Do đó:

- vị trí cuối `≤ S_k` và cùng tính chẵn/lẻ với `S_k`.

**Quan sát B (điều kiện đủ).** **Mọi** số nguyên `F` trong `[0, S_k]` đều là tổng của một tập con của `{1, ..., k}`. (Chứng minh quy nạp ở mục 10.) Do đó mọi vị trí `S_k − 2F` với `0 ≤ F ≤ S_k`, tức mọi vị trí **cùng tính chẵn/lẻ với `S_k`** trong `[−S_k, S_k]`, đều đạt được.

**Kết luận:** với `T > 0`, `k` bước đủ để tới `T` ⟺ `S_k ≥ T` và `S_k ≡ T (mod 2)`. Đáp án là `k` **nhỏ nhất** thoả mãn. Chỉ cần tăng `k` cho tới khi thoả: số lần lặp cỡ `√(2T)`, tối đa khoảng `44 723` khi `T = 10^9`.

Cell dưới kiểm chứng Quan sát B bằng code: với mọi `k ≤ 12`, mọi `F ∈ [0, S_k]` đều là tổng một tập con của `{1..k}`.

In [9]:
def subset_sums(k):
    sums = {0}
    for i in range(1, k + 1):
        sums |= {s + i for s in sums}
    return sums

for k in range(0, 13):
    S = k * (k + 1) // 2
    assert subset_sums(k) == set(range(S + 1)), k
print("với mọi k <= 12: tổng các tập con của {1..k} phủ kín 0..S_k")

với mọi k <= 12: tổng các tập con của {1..k} phủ kín 0..S_k


## 7. Các cách tiếp theo

### 7.1. Cách 2 — BFS theo tầng: tập các vị trí đạt được

**Giải thích từng đoạn.**

1. `positions = {0}` (sau 0 bước chỉ ở `0`).
2. Ở bước `k = 1, 2, ...`: `positions = { p + k, p − k for p in positions }` — vị trí mới là vị trí cũ cộng hoặc trừ `k`. Dùng **`set`** nên các dãy khác nhau cho cùng vị trí được gộp lại.
3. Khi `T ∈ positions` thì `k` là đáp án.

Kích thước `positions` tối đa cỡ `S_k` (các vị trí cùng tính chẵn/lẻ), tức `O(k^2)`, thay vì `2^k` như Cách 1. Vẫn chậm với `T` lớn nhưng chạy tốt ở `T` cỡ vài nghìn.

**Code demo:**

In [10]:
def reach_positions(target):
    T = abs(target)
    positions = {0}
    k = 0
    while T not in positions:
        k += 1
        positions = {p + k for p in positions} | {p - k for p in positions}
    return k

for target, expected in examples:
    print(reach_positions(target), expected)
print([reach_positions(t) for t in range(1, 13)])
print(reach_positions(1000), reach_positions(-1000))

3 3
2 2
[1, 3, 2, 3, 5, 3, 5, 4, 5, 4, 5, 7]
47 47


**Code nộp LeetCode (Cách 2):** (đúng nhưng chỉ chạy nhanh với `|target|` nhỏ; với `10^9` sẽ quá chậm.)

In [11]:
class SolutionPositions:
    def reachNumber(self, target: int) -> int:
        T = abs(target)
        positions = {0}
        k = 0
        while T not in positions:
            k += 1
            positions = {p + k for p in positions} | {p - k for p in positions}
        return k

print(SolutionPositions().reachNumber(3))   # 2

2


**Độ phức tạp (Cách 2):** Time `O(k · |positions|) = O(k^3)` với `k ≈ √(2T)`, tức `O(T^{1.5})`; Space `O(k^2)`. Với `T = 10^9` không khả thi.

### 7.2. Cách 3 — Tăng `k` đến khi tổng đủ lớn và cùng tính chẵn/lẻ (lời giải nộp)

**Giải thích từng đoạn.**

1. `T = abs(target)` (đối xứng).
2. `k = 0`, `s = 0`. Lặp: `k += 1`, `s += k` cho đến khi **`s ≥ T`** và **`(s − T) % 2 == 0`**.
3. Trả về `k`.

Mỗi lần lặp thêm bước `k` vào tổng `s = S_k`. Điều kiện dừng đúng bằng kết luận ở mục 6.

**Code demo:**

In [12]:
def reach_linear(target):
    T = abs(target)
    k = s = 0
    while s < T or (s - T) % 2 != 0:
        k += 1
        s += k
    return k

for target, expected in examples:
    print(reach_linear(target), expected)
print([reach_linear(t) for t in range(1, 13)])
print(reach_linear(10 ** 9), reach_linear(-(10 ** 9)))

3 3
2 2
[1, 3, 2, 3, 5, 3, 5, 4, 5, 4, 5, 7]
44723 44723


**Code nộp LeetCode (Cách 3):**

In [13]:
class SolutionLinear:
    def reachNumber(self, target: int) -> int:
        target = abs(target)
        k = s = 0
        while s < target or (s - target) % 2:
            k += 1
            s += k
        return k

print(SolutionLinear().reachNumber(2))   # 3

3


**Độ phức tạp (Cách 3):** Time `O(√T)` (khoảng `44 723` vòng lặp khi `T = 10^9`), Space `O(1)`.

### 7.3. Cách 4 — Tìm kiếm nhị phân cho `k` nhỏ nhất với `S_k ≥ T`, rồi sửa chẵn/lẻ

**Giải thích từng đoạn.**

1. `S_k = k(k+1)/2` **tăng nghiêm ngặt** theo `k`. Vậy `{k : S_k ≥ T}` là một đoạn từ `k0` trở đi. Nhị phân trên `[0, 2·10^5]` (hoặc `[1, T]`) để tìm `k0` nhỏ nhất.
2. Từ `k0`, nếu `(S_{k0} − T)` lẻ thì tăng `k` đến khi chẵn: chỉ cần tối đa **2** lần tăng (xem mục 10).
3. Trả về `k`.

**Code demo:**

In [14]:
def reach_binary(target):
    T = abs(target)
    lo, hi = 0, 200000            # hi đủ lớn: S_200000 = 2*10^10 > 10^9
    while lo < hi:
        mid = (lo + hi) // 2
        if mid * (mid + 1) // 2 >= T:
            hi = mid              # mid thỏa, thử nhỏ hơn
        else:
            lo = mid + 1          # mid chưa đủ
    k = lo
    while (k * (k + 1) // 2 - T) % 2:
        k += 1                    # tối đa 2 lần
    return k

for target, expected in examples:
    print(reach_binary(target), expected)
print([reach_binary(t) for t in range(1, 13)])
print(reach_binary(10 ** 9), reach_binary(-(10 ** 9)))

3 3
2 2
[1, 3, 2, 3, 5, 3, 5, 4, 5, 4, 5, 7]
44723 44723


**Code nộp LeetCode (Cách 4):**

In [15]:
class SolutionBinary:
    def reachNumber(self, target: int) -> int:
        T = abs(target)
        lo, hi = 0, 200000
        while lo < hi:
            mid = (lo + hi) // 2
            if mid * (mid + 1) // 2 >= T:
                hi = mid
            else:
                lo = mid + 1
        k = lo
        while (k * (k + 1) // 2 - T) % 2:
            k += 1
        return k

print(SolutionBinary().reachNumber(3))   # 2

2


**Độ phức tạp (Cách 4):** Time `O(log T)` cho nhị phân (+ tối đa 2 bước sửa), Space `O(1)`.

### 7.4. Cách 5 — Công thức đóng bằng `isqrt`

**Giải thích từng đoạn.**

1. `k` lớn nhất với `S_k ≤ T` thoả `(2k+1)^2 ≤ 8T + 1`, tức `k = (isqrt(8T + 1) − 1) // 2`.
2. Nếu `S_k < T` thì `k += 1` (để `S_k ≥ T`): đây là `k0`.
3. Sửa chẵn/lẻ: `while (S_k − T) % 2: k += 1` (tối đa 2 lần).

Không có vòng lặp theo `T`: thời gian hằng số (với số nguyên cỡ `10^9`).

**Code demo:**

In [16]:
def reach_isqrt(target):
    T = abs(target)
    k = (isqrt(8 * T + 1) - 1) // 2          # k lớn nhất với S_k <= T
    if k * (k + 1) // 2 < T:
        k += 1                               # k nhỏ nhất với S_k >= T
    while (k * (k + 1) // 2 - T) % 2:
        k += 1                               # tối đa 2 lần
    return k

for target, expected in examples:
    print(reach_isqrt(target), expected)
print([reach_isqrt(t) for t in range(1, 13)])
print(reach_isqrt(10 ** 9), reach_isqrt(-(10 ** 9)))

3 3
2 2
[1, 3, 2, 3, 5, 3, 5, 4, 5, 4, 5, 7]
44723 44723


**Code nộp LeetCode (Cách 5):**

In [17]:
class SolutionMath:
    def reachNumber(self, target: int) -> int:
        T = abs(target)
        k = (isqrt(8 * T + 1) - 1) // 2
        if k * (k + 1) // 2 < T:
            k += 1
        while (k * (k + 1) // 2 - T) % 2:
            k += 1
        return k

print(SolutionMath().reachNumber(2))   # 3

3


**Độ phức tạp (Cách 5):** Time `O(1)` (một phép `isqrt` và tối đa 3 lần lặp), Space `O(1)`. (Trong LeetCode cần `from math import isqrt`.)

## 8. TRACE từng bước

Trace Cách 3 cho `target = 5`:

| k | s = S_k | s ≥ T? | s − T | chẵn? | dừng? |
|---|---|---|---|---|---|
| 1 | 1 | không | −4 | — | không |
| 2 | 3 | không | −2 | — | không |
| 3 | 6 | có | 1 | **lẻ** | không |
| 4 | 10 | có | 5 | **lẻ** | không |
| 5 | 15 | có | 10 | chẵn | **dừng, đáp án 5** |

Chú ý `k = 3` đã đủ lớn nhưng lệch chẵn/lẻ; `k = 4` vẫn lệch (vì thêm bước `4` chẵn không đổi tính chẵn/lẻ của tổng); phải tới `k = 5` — **thêm 2 bước**.

Hàm `trace_reach` in bảng đó cho `target` bất kỳ, kèm cách dựng đường đi cụ thể: bước nào đổi sang trái.

In [18]:
def trace_reach(target):
    T = abs(target)
    print(f"target = {target} -> T = |target| = {T}")
    print(f"{'k':>3} {'s=S_k':>7} {'s>=T':>6} {'s-T':>6} {'chẵn?':>7}")
    k = s = 0
    while s < T or (s - T) % 2:
        k += 1
        s += k
        print(f"{k:>3} {s:>7} {str(s >= T):>6} {s - T:>6} {str((s - T) % 2 == 0) if s >= T else '—':>7}")
    print(f"k = {k}: s = {s}, s - T = {s - T}, F = (s - T) / 2 = {(s - T) // 2}")
    return k

trace_reach(5)

target = 5 -> T = |target| = 5
  k   s=S_k   s>=T    s-T   chẵn?
  1       1  False     -4       —
  2       3  False     -2       —
  3       6   True      1   False
  4      10   True      5   False
  5      15   True     10    True
k = 5: s = 15, s - T = 10, F = (s - T) / 2 = 5


5

Phần còn lại của trace: **dựng đường đi**. Cần đổi sang trái các bước có tổng `F = (S_k − T) / 2`. Tham lam từ bước lớn nhất xuống: nếu `i ≤ F` thì đổi bước `i` sang trái và `F -= i`. Với `T = 5, k = 5, S = 15, F = 5`: bước `5 ≤ 5` → đổi bước `5`, `F = 0`. Đường đi: `+1 +2 +3 +4 −5 = 5` ✓.

In [19]:
def build_path(target):
    T = abs(target)
    k = reach_linear(T)
    s = k * (k + 1) // 2
    F = (s - T) // 2
    signs = [1] * k                       # signs[i-1] là hướng ở lần i
    for i in range(k, 0, -1):             # tham lam: đổi bước lớn nhất có thể
        if i <= F:
            signs[i - 1] = -1
            F -= i
    assert F == 0
    if target < 0:
        signs = [-x for x in signs]       # đối xứng
    return signs

for t in (5, 2, 3, -5, 7):
    sg = build_path(t)
    pos = 0
    trail = [0]
    for i, d in enumerate(sg, start=1):
        pos += d * i
        trail.append(pos)
    print(f"target = {t:>2}: hướng = {['+' if d > 0 else '-' for d in sg]} | đường đi {trail}")
    assert pos == t

target =  5: hướng = ['+', '+', '+', '+', '-'] | đường đi [0, 1, 3, 6, 10, 5]
target =  2: hướng = ['+', '-', '+'] | đường đi [0, 1, -1, 2]
target =  3: hướng = ['+', '+'] | đường đi [0, 1, 3]
target = -5: hướng = ['-', '-', '-', '-', '+'] | đường đi [0, -1, -3, -6, -10, -5]
target =  7: hướng = ['+', '+', '+', '-', '+'] | đường đi [0, 1, 3, 6, 2, 7]


Trace thứ hai: hai tham số hay bị nhầm — khi nào cần thêm **0, 1 hoặc 2** bước sau lần đầu `S_k ≥ T`? Ta in cho `T = 1..20`: `k0` (nhỏ nhất với `S_k ≥ T`), `k` đáp án và số bước thêm.

In [20]:
print(f"{'T':>3} {'k0':>3} {'S_k0':>5} {'đáp án':>7} {'thêm':>5}")
for T in range(1, 21):
    k0 = 0
    while k0 * (k0 + 1) // 2 < T:
        k0 += 1
    ans = reach_linear(T)
    print(f"{T:>3} {k0:>3} {k0 * (k0 + 1) // 2:>5} {ans:>7} {ans - k0:>5}")

  T  k0  S_k0  đáp án  thêm
  1   1     1       1     0
  2   2     3       3     1
  3   2     3       2     0
  4   3     6       3     0
  5   3     6       5     2
  6   3     6       3     0
  7   4    10       5     1
  8   4    10       4     0
  9   4    10       5     1
 10   4    10       4     0
 11   5    15       5     0
 12   5    15       7     2
 13   5    15       5     0
 14   5    15       7     2
 15   5    15       5     0
 16   6    21       7     1
 17   6    21       6     0
 18   6    21       7     1
 19   6    21       6     0
 20   6    21       7     1


## 9. Code nộp LeetCode

Lời giải chuẩn: vòng lặp tăng `k` (Cách 3) — ngắn, `O(√T)` (≤ 44 723 vòng), không cần nhớ công thức. (Cách nhị phân/`isqrt` nhanh hơn nhưng không cần cho giới hạn này.) Đúng `class Solution` và signature.

In [21]:
from typing import List

class Solution:
    def reachNumber(self, target: int) -> int:
        target = abs(target)             # đối xứng: -T cũng cần số bước như T
        k = s = 0                        # s = 1 + 2 + ... + k
        while s < target or (s - target) % 2:
            k += 1
            s += k
        return k

sol = Solution()
for target, expected in examples:
    got = sol.reachNumber(target)
    print(got, expected, "OK" if got == expected else "SAI")

3 3 OK
2 2 OK


## 10. Vì sao đúng

Đặt `T = |target| ≥ 1` và `S_k = k(k+1)/2`.

**Mệnh đề.** Có thể tới `T` bằng đúng `k` lần di chuyển ⟺ `S_k ≥ T` và `S_k ≡ T (mod 2)`.

**(⇒) Điều kiện cần.** Gọi `F` là tập các bước đi sang trái. Vị trí cuối `= S_k − 2·sum(F)`. Đặt bằng `T`: `S_k − T = 2·sum(F) ≥ 0`, do đó `S_k ≥ T` và `S_k − T` chẵn.

**(⇐) Điều kiện đủ.** Cho `S_k ≥ T`, `S_k − T` chẵn. Đặt `d = (S_k − T) / 2`, là số nguyên với `0 ≤ d ≤ S_k / 2 ≤ S_k`. **Bổ đề:** mọi số nguyên `d ∈ [0, S_k]` là tổng một tập con của `{1, ..., k}`.

*Chứng minh bổ đề bằng quy nạp theo `k`.* `k = 0`: chỉ có `d = 0` (tập rỗng). Giả sử đúng với `k`; với `k + 1`: nếu `d ≤ S_k`, dùng tập con của `{1..k}`; nếu `d ∈ [k+1, S_{k+1}]` thì `d − (k+1) ∈ [0, S_k]` (vì `S_{k+1} − (k+1) = S_k`), dùng tập con cho `d − (k+1)` rồi thêm phần tử `k + 1`. Hai đoạn `[0, S_k]` và `[k+1, S_{k+1}]` phủ kín `[0, S_{k+1}]` vì `k + 1 ≤ S_k + 1` (đúng với mọi `k ≥ 0`). ∎

Lấy tập con `F` có tổng `d` làm tập bước đi sang trái: vị trí cuối `= S_k − 2d = T`. ✓

**Đáp án là `k` nhỏ nhất thoả mệnh đề** (vì mệnh đề là điều kiện cần và đủ cho từng `k`, và code kiểm tra `k` tăng dần).

**Thêm tối đa 2 bước.** Gọi `k0` là `k` nhỏ nhất với `S_k ≥ T`. Nếu `S_{k0} − T` lẻ: `S_{k0+1} = S_{k0} + (k0 + 1)`; nếu `k0 + 1` lẻ thì tính chẵn/lẻ đổi → đáp án `k0 + 1`. Nếu `k0 + 1` chẵn thì `S_{k0+1}` vẫn lệch, nhưng `S_{k0+2} = S_{k0+1} + (k0 + 2)` với `k0 + 2` lẻ → chẵn/lẻ đổi → đáp án `k0 + 2`. Vậy luôn `≤ k0 + 2`.

Cell dưới kiểm chứng: (a) mệnh đề trên với `k ≤ 11` bằng cách liệt kê mọi vị trí; (b) đáp án trùng oracle BFS-tập-vị trí cho `1 ≤ T ≤ 400`; (c) số bước thêm ∈ {0, 1, 2} cho `T ≤ 10^5`.

In [22]:
# (a) mệnh đề: tập vị trí đạt được sau đúng k bước = các số cùng chẵn/lẻ với S_k trong [-S_k, S_k]
positions = {0}
for k in range(1, 12):
    positions = {p + k for p in positions} | {p - k for p in positions}
    S = k * (k + 1) // 2
    assert positions == {x for x in range(-S, S + 1) if (x - S) % 2 == 0}, k
print("(a) mệnh đề đúng với mọi k <= 11")

# (b) khớp oracle BFS-tập-vị trí
for T in range(1, 401):
    assert reach_linear(T) == reach_positions(T) == reach_binary(T) == reach_isqrt(T), T
print("(b) 4 cách khớp nhau cho T = 1..400")

# (c) số bước thêm tối đa 2
max_extra = 0
for T in range(1, 100001):
    k0 = (isqrt(8 * T + 1) - 1) // 2
    if k0 * (k0 + 1) // 2 < T:
        k0 += 1
    max_extra = max(max_extra, reach_isqrt(T) - k0)
print("(c) số bước thêm lớn nhất:", max_extra)
assert max_extra == 2

(a) mệnh đề đúng với mọi k <= 11


(b) 4 cách khớp nhau cho T = 1..400


(c) số bước thêm lớn nhất: 2


## 11. Time / Space Complexity

- **Cách chuẩn (Cách 3):** Time `O(√T)` — `k` tăng đến cỡ `√(2T)`; với `T = 10^9` là `~ 44 723` vòng; Space `O(1)`.
- **Nhị phân (Cách 4):** `O(log T)`; **`isqrt` (Cách 5):** `O(1)`.
- Vét cạn dấu `O(2^k · k)`, BFS tập vị trí `O(T^{1.5})`: chỉ dùng cho `T` nhỏ.

Đo số vòng lặp thực tế của Cách 3 ở vài `T`:

In [23]:
def loops_linear(target):
    T = abs(target)
    k = s = 0
    while s < T or (s - T) % 2:
        k += 1
        s += k
    return k

for T in (10, 1000, 10 ** 6, 10 ** 9):
    print(f"T = {T:>10}: k = {loops_linear(T):>6} vòng lặp; so với √(2T) = {int((2 * T) ** 0.5)}")

T =         10: k =      4 vòng lặp; so với √(2T) = 4
T =       1000: k =     47 vòng lặp; so với √(2T) = 44
T =    1000000: k =   1415 vòng lặp; so với √(2T) = 1414
T = 1000000000: k =  44723 vòng lặp; so với √(2T) = 44721


In [24]:
# Cho thấy BFS-tập-vị trí phình to (không dùng được với T lớn), còn công thức vẫn tức thì
sizes = []
positions = {0}
for k in range(1, 61):
    positions = {p + k for p in positions} | {p - k for p in positions}
    if k in (10, 20, 40, 60):
        sizes.append((k, len(positions)))
print("k và |positions|:", sizes)           # ~ S_k phần tử: tăng bậc hai theo k
print("T = 10^9 cần k =", reach_isqrt(10 ** 9), "=> |positions| cỡ", reach_isqrt(10 ** 9) ** 2 // 2, "(quá lớn)")

k và |positions|: [(10, 56), (20, 211), (40, 821), (60, 1831)]
T = 10^9 cần k = 44723 => |positions| cỡ 1000073364 (quá lớn)


## 12. Bộ test

Gồm: ví dụ đề, edge case, phản ví dụ cho các cách sai, đối chiếu các cách bằng `assert` (kèm oracle độc lập), và test ngẫu nhiên có seed đúng ràng buộc.

In [25]:
sol = Solution()

# 1) Ví dụ đề
assert sol.reachNumber(2) == 3
assert sol.reachNumber(3) == 2
print("ví dụ đề OK")

# 2) Edge case
assert sol.reachNumber(1) == 1
assert sol.reachNumber(-1) == 1
assert sol.reachNumber(-2) == 3                   # đối xứng
assert sol.reachNumber(-3) == 2
assert sol.reachNumber(4) == 3                    # 1 - 2 + ... : +1 -2... cụ thể: -1 +2 +3 = 4
assert sol.reachNumber(5) == 5                    # cần thêm 2 bước sau k0 = 3
assert sol.reachNumber(6) == 3                    # S_3 = 6
assert sol.reachNumber(10) == 4                   # S_4 = 10
assert sol.reachNumber(1000000000) == 44723       # cực đại dương theo đề
assert sol.reachNumber(-1000000000) == 44723
assert sol.reachNumber(999999999) == reach_isqrt(999999999)
print("edge case OK; reachNumber(10^9) =", sol.reachNumber(10 ** 9))

ví dụ đề OK
edge case OK; reachNumber(10^9) = 44723


In [26]:
# 3) Mọi cách + oracle độc lập (BFS-tập-vị trí & vét cạn dấu) trên dải nhỏ
classes = [SolutionPositions(), SolutionLinear(), SolutionBinary(), SolutionMath(), Solution()]
for T in range(1, 301):
    expected = reach_positions(T)                # oracle: tập vị trí
    for c in classes:
        assert c.reachNumber(T) == expected, (type(c).__name__, T)
        assert c.reachNumber(-T) == expected, (type(c).__name__, -T)
for T in range(1, 31):
    assert reach_brute(T) == reach_positions(T), T   # vét cạn dấu (k tới ~8)
print("T = ±1..300: 5 cách khớp oracle tập vị trí; vét cạn dấu khớp tới 30")

# 4) Ngẫu nhiên có seed, đúng ràng buộc |target| <= 10^9, target != 0
random.seed(2026)
fast = [SolutionLinear(), SolutionBinary(), SolutionMath(), Solution()]
for _ in range(300):
    t = random.randint(1, 10 ** 9) * random.choice((-1, 1))
    vals = {type(c).__name__: c.reachNumber(t) for c in fast}
    assert len(set(vals.values())) == 1, (t, vals)
for _ in range(300):                              # dải trung bình: so với oracle tập vị trí
    t = random.randint(1, 1500) * random.choice((-1, 1))
    assert sol.reachNumber(t) == reach_positions(t), t
print("300 + 300 input ngẫu nhiên: khớp")

# 5) Test tính chất: kết quả thoả mệnh đề (S_k >= T, cùng chẵn/lẻ) và k - 1 KHÔNG thoả
random.seed(7)
for _ in range(300):
    T = random.randint(1, 10 ** 9)
    k = sol.reachNumber(T)
    S = k * (k + 1) // 2
    assert S >= T and (S - T) % 2 == 0
    km = k - 1
    Sm = km * (km + 1) // 2
    assert not (Sm >= T and (Sm - T) % 2 == 0)
    # tính tối ưu: mọi k' < k không thoả (kiểm tra vài giá trị)
    for kk in range(max(1, k - 6), k):
        S2 = kk * (kk + 1) // 2
        assert not (S2 >= T and (S2 - T) % 2 == 0)
print("test tính chất OK")

T = ±1..300: 5 cách khớp oracle tập vị trí; vét cạn dấu khớp tới 30


300 + 300 input ngẫu nhiên: khớp


test tính chất OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên lấy `abs(target)`

Với `target` âm, điều kiện `s < target` luôn sai ngay từ đầu, nên vòng lặp chỉ còn kiểm tra tính chẵn/lẻ và dừng sai.

In [27]:
def wrong_no_abs(target):
    k = s = 0
    while s < target or (s - target) % 2:          # SAI: không dùng abs
        k += 1
        s += k
    return k

print("sai :", wrong_no_abs(-3), "(kỳ vọng 2)")
print("đúng:", Solution().reachNumber(-3))
print("sai :", wrong_no_abs(-2), "| đúng:", Solution().reachNumber(-2))

sai : 1 (kỳ vọng 2)
đúng: 2
sai : 0 | đúng: 3


**Vì sao sai:** với `target = -3`, `s = 0 < -3` sai; `(0 − (−3)) % 2 = 1` nên vẫn lặp; `k = 1, s = 1`: `(1 + 3) % 2 = 0` → dừng với `k = 1`, trong khi thực tế phải đi tới `-3` với 2 bước (`-1 - 2`). **Quy tắc:** đối xứng → `abs` ngay đầu hàm.

### Lỗi 2 — Chỉ kiểm tra `S_k ≥ T`, quên tính chẵn/lẻ

Nghĩ "đủ xa là tới được", nhưng đi sai hướng làm tổng giảm **số chẵn**, nên không thể "lùi lại 1 đơn vị".

In [28]:
def wrong_no_parity(target):
    T = abs(target)
    k = s = 0
    while s < T:                                    # SAI: thiếu điều kiện chẵn/lẻ
        k += 1
        s += k
    return k

for t in (2, 5, 7):
    print(f"target = {t}: sai {wrong_no_parity(t)} | đúng {Solution().reachNumber(t)}")

target = 2: sai 2 | đúng 3
target = 5: sai 3 | đúng 5
target = 7: sai 4 | đúng 5


**Vì sao sai:** `target = 2`: `S_2 = 3 ≥ 2` nhưng cần "lùi" 1 đơn vị — không có bước nào làm tổng giảm số lẻ (đổi hướng bước `i` giảm `2i`). **Quy tắc:** `(S_k − T)` phải **chẵn**.

### Lỗi 3 — Cho rằng lệch chẵn/lẻ thì chỉ cần thêm đúng 1 bước

Thêm bước `k + 1` đổi tính chẵn/lẻ của tổng **chỉ khi `k + 1` lẻ**. Khi `k + 1` chẵn, phải thêm 2 bước.

In [29]:
def wrong_plus_one(target):
    T = abs(target)
    k = s = 0
    while s < T:
        k += 1
        s += k
    if (s - T) % 2:
        k += 1                                      # SAI: tưởng chỉ cần thêm 1
    return k

bad = [t for t in range(1, 60) if wrong_plus_one(t) != Solution().reachNumber(t)]
print("các target sai:", bad[:10], "...  tổng", len(bad))
print("target = 5:", wrong_plus_one(5), "(đúng 5)")

các target sai: [5, 12, 14, 23, 25, 27, 38, 40, 42, 44] ...  tổng 12
target = 5: 4 (đúng 5)


**Vì sao sai:** với `T = 5`, `k0 = 3` (`S = 6`, lệch lẻ). Thêm bước `4` (chẵn) cho `S = 10`, hiệu `5` vẫn lẻ. Phải thêm bước `5` (lẻ) mới đổi được. **Quy tắc:** cứ để vòng `while` tự kiểm tra, không đoán số bước thêm.

### Lỗi 4 — Tham lam mô phỏng "đi tới đích, nếu vượt thì quay lại"

Nghe hợp lý nhưng không tối ưu và có thể chạy mãi.

In [30]:
def wrong_greedy_walk(target, limit=10000):
    pos, step = 0, 0
    while pos != target and step < limit:
        step += 1
        pos += step if pos < target else -step        # SAI: tham lam theo vị trí hiện tại
    return step if pos == target else None

for t in (2, 3, 4, 5):
    print(f"target = {t}: tham lam {wrong_greedy_walk(t)} | đúng {Solution().reachNumber(t)}")
bad = [t for t in range(1, 40) if wrong_greedy_walk(t) != Solution().reachNumber(t)]
print("số target sai trong 1..39:", len(bad))

target = 2: tham lam None | đúng 3
target = 3: tham lam 2 | đúng 2
target = 4: tham lam None | đúng 3
target = 5: tham lam None | đúng 5
số target sai trong 1..39: 31


**Vì sao sai:** quyết định dấu theo vị trí hiện tại là một quy tắc cục bộ; nó có thể dao động quanh đích mà không chạm đúng. Bài toán thật sự là **tổng có dấu** của `1..k`, không phải chuỗi quyết định theo vị trí.

### Lỗi 5 — Dùng vét cạn dấu/BFS cho `target` lớn

`target = 10^9` cần `k = 44 723`; `2^{44723}` dãy dấu hoặc tập vị trí cỡ `10^9` phần tử là bất khả thi. Hãy **nhìn ra quy luật** thay vì liệt kê.

In [31]:
import time
t0 = time.perf_counter(); r = reach_positions(2000); t1 = time.perf_counter()
r2 = Solution().reachNumber(2000); t2 = time.perf_counter()
print("cùng đáp án:", r == r2, "| tập vị trí chậm hơn công thức:", (t1 - t0) > (t2 - t1))

cùng đáp án: True | tập vị trí chậm hơn công thức: True


**Cách nhớ:** khi giới hạn đầu vào lên `10^9`, đáp án phải là công thức hoặc vòng lặp `O(√n)`, `O(log n)`.

## 14. Mẹo & nhận dạng pattern

- **"Đi `1, 2, 3, ...` bước, mỗi bước chọn trái/phải"** → tổng có dấu của `1..k`. Quan sát: đổi dấu số `i` làm tổng giảm `2i` ⇒ **bất biến chẵn/lẻ**.
- **Bất biến chẵn/lẻ** là công cụ mạnh: hai quan sát "tổng ≥ T" và "cùng chẵn/lẻ" thường quyết định bài toán.
- **Mọi số từ `0` đến `S_k` là tổng con của `{1..k}`** — bổ đề quen thuộc, dùng để chuyển "cần" thành "đủ".
- **Số tam giác & giải bất phương trình bậc hai:** dùng `isqrt`, hoặc nhị phân, hoặc đơn giản là vòng lặp `O(√n)`.
- **Đối xứng:** `target` âm ↔ dương, nên `abs` đầu tiên.
- **Kiểm chứng:** viết oracle vét cạn (tập vị trí) cho `T` nhỏ rồi so sánh; in bảng kết quả nhỏ để thấy quy luật.
- Bài họ hàng: Minimum Moves to Reach Target Score (2139), Jump Game, Super Washing Machines, Can I Win (464), Partition Equal Subset Sum (416; tổng con).

## 15. Biến thể

### 15.1. Trả về dãy hướng đi cụ thể

Sau khi có `k`, đặt `F = (S_k − T) / 2` rồi **tham lam từ bước lớn nhất**: nếu `i ≤ F` thì đổi bước `i` sang trái và `F -= i`. Chạy đúng vì bổ đề: tham lam trên `k, k−1, ..., 1` luôn cho `F = 0` khi `0 ≤ F ≤ S_k` (xem chứng minh bằng quy nạp tương tự mục 10). Mục 8 đã dựng `build_path`; ta kiểm chứng hàng loạt.

In [32]:
for t in list(range(-300, 301)):
    if t == 0:
        continue
    sg = build_path(t)
    assert len(sg) == Solution().reachNumber(t)
    assert sum(d * i for d, i in zip(sg, range(1, len(sg) + 1))) == t
print("build_path đúng cho mọi target từ -300 đến 300 (khác 0)")
print(build_path(10 ** 9)[-5:], len(build_path(10 ** 9)))

build_path đúng cho mọi target từ -300 đến 300 (khác 0)
[1, 1, 1, 1, -1] 44723


### 15.2. Cho phép "đứng yên" ở một lần di chuyển

Giả sử ở lần `i` ta được chọn: đi `+i`, `-i`, hoặc **không đi** (`0`). Khi đó mọi vị trí trong `[−S_k, S_k]` đều đạt được (không cần cùng chẵn/lẻ), nên đáp án là `k` nhỏ nhất với **`S_k ≥ T`** — bỏ điều kiện chẵn/lẻ. Cell dưới kiểm chứng bằng BFS tập vị trí với ba lựa chọn.

In [33]:
def reach_with_stay_formula(target):
    T = abs(target)
    k = s = 0
    while s < T:
        k += 1
        s += k
    return k

def reach_with_stay_bfs(target):
    T = abs(target)
    positions = {0}
    k = 0
    while T not in positions:
        k += 1
        positions = {p + d * k for p in positions for d in (-1, 0, 1)}
    return k

for T in range(1, 200):
    assert reach_with_stay_formula(T) == reach_with_stay_bfs(T), T
print("có lựa chọn đứng yên: đáp án = k nhỏ nhất với S_k >= T")
print([reach_with_stay_formula(T) for T in range(1, 13)], "so với", [Solution().reachNumber(T) for T in range(1, 13)])

có lựa chọn đứng yên: đáp án = k nhỏ nhất với S_k >= T
[1, 2, 2, 3, 3, 3, 4, 4, 4, 4, 5, 5] so với [1, 3, 2, 3, 5, 3, 5, 4, 5, 4, 5, 7]


### 15.3. Xuất phát không phải từ 0

Nếu xuất phát từ `p0` và đích `target`, chỉ cần dùng `|target − p0|` (dịch gốc toạ độ). Đây là cách tổng quát hoá không đổi thuật toán.

### 15.4. Dạng đóng theo `k mod 4`

Vì `S_k` chẵn khi `k ≡ 0, 3 (mod 4)` và lẻ khi `k ≡ 1, 2 (mod 4)`, ta có thể thay vòng `while` bằng bảng: sau khi tìm `k0` (nhỏ nhất với `S_k ≥ T`), nếu `S_{k0} ≡ T (mod 2)` thì trả `k0`; ngược lại trả `k0 + 1` nếu `k0` chẵn (vì `k0 + 1` lẻ), trả `k0 + 2` nếu `k0` lẻ (vì `k0 + 1` chẵn). Kiểm chứng:

In [34]:
def reach_by_table(target):
    T = abs(target)
    k0 = (isqrt(8 * T + 1) - 1) // 2
    if k0 * (k0 + 1) // 2 < T:
        k0 += 1
    if (k0 * (k0 + 1) // 2 - T) % 2 == 0:
        return k0
    return k0 + 1 if k0 % 2 == 0 else k0 + 2

for T in range(1, 50001):
    assert reach_by_table(T) == reach_isqrt(T), T
print("bảng chẵn/lẻ theo k0 khớp cách isqrt cho T = 1..50000")

bảng chẵn/lẻ theo k0 khớp cách isqrt cho T = 1..50000


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn dấu | Thử mọi dãy `±` cho từng `k` | `O(2^k · k)` | `O(k)` | Chỉ để kiểm tra `T ≤ ~30` |
| 2. BFS tập vị trí | `set` các vị trí sau mỗi bước | `O(T^{1.5})` | `O(T)` | Oracle `T` đến vài nghìn |
| 3. Tăng `k` đến khi đủ & chẵn/lẻ | `S_k ≥ T` và `(S_k − T)` chẵn | `O(√T)` | `O(1)` | **Lời giải chuẩn nộp LeetCode** |
| 4. Nhị phân | Tìm `k0` bằng nhị phân, sửa chẵn/lẻ ≤ 2 lần | `O(log T)` | `O(1)` | Khi `T` rất lớn hơn |
| 5. Công thức `isqrt` | `k0 = ⌈(√(8T+1) − 1)/2⌉` | `O(1)` | `O(1)` | Ngắn nhất; cần cẩn thận làm tròn |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy kiểm tra:

```python
print(Solution().reachNumber(2))
print(Solution().reachNumber(-3))
print(Solution().reachNumber(5))
```

In [35]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `target = 4`: tìm `k` nhỏ nhất thoả mãn, tính `S_k`, `F = (S_k − 4) / 2`, rồi cho biết **bước nào** đi sang trái. Điền vào các biến.

In [36]:
# Bài 2: điền kết quả tính tay
hand_k = None          # số lần di chuyển ít nhất
hand_left_steps = None # list các bước (giá trị i) đi sang trái, ví dụ [2]

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành Cách 3.

In [37]:
def exercise_3_reach_number(target: int) -> int:
    # TODO 1: target = abs(target)
    # TODO 2: k = 0, s = 0
    # TODO 3: lặp khi s < target HOẶC (s - target) lẻ:  k += 1; s += k
    # TODO 4: return k
    return None


# assert exercise_3_reach_number(2) == 3
# assert exercise_3_reach_number(3) == 2
# assert exercise_3_reach_number(-5) == 5
# assert exercise_3_reach_number(10 ** 9) == 44723

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 5, hãy tự viết `exercise_4_isqrt(target)`: tính `k0` bằng `isqrt(8T + 1)`, rồi sửa chẵn/lẻ.

In [38]:
def exercise_4_isqrt(target: int) -> int:
    # TODO
    pass


# assert exercise_4_isqrt(2) == 3
# assert exercise_4_isqrt(3) == 2
# assert exercise_4_isqrt(-10 ** 9) == 44723
# assert exercise_4_isqrt(5) == 5

### Bài 5 — Mini challenge: trả về dãy hướng đi (khó)

Viết `exercise_5_signs(target)` trả về **danh sách hướng** (`+1` phải, `-1` trái) có độ dài `numMoves` tối thiểu sao cho `sum(d * i for i, d in enumerate(signs, 1)) == target`. Gợi ý: tính `k`, `F = (S_k − T) / 2`, tham lam từ bước lớn nhất; nếu `target < 0` thì đổi dấu toàn bộ.

```python
exercise_5_signs(2)   -> [1, -1, 1]       # +1 -2 +3 = 2 (một đáp án hợp lệ; có thể khác)
exercise_5_signs(-3)  -> [-1, -1]
```

In [39]:
def exercise_5_signs(target: int) -> List[int]:
    # TODO
    pass


# kết quả được kiểm tra ở mục 19 bằng cách tính tổng có dấu

## 18. Gợi ý 3 mức

**Bài 3 / Bài 4**

- *Mức 1 (nhẹ):* Đi sang trái ở bước `i` làm tổng giảm `2i`. Từ đó tổng cuối luôn cùng tính chẵn/lẻ với `1 + 2 + ... + k`.
- *Mức 2 (vừa):* Điều kiện đủ để tới `T` với `k` bước: `S_k ≥ T` và `S_k − T` chẵn. Chỉ cần tìm `k` nhỏ nhất.
- *Mức 3 (gần đáp án):* Vòng lặp: `while s < T or (s - T) % 2: k += 1; s += k`. Với `isqrt`: `k = (isqrt(8T + 1) − 1) // 2`, nếu `S_k < T` thì `k += 1`, rồi `while (S_k − T) % 2: k += 1`.

**Bài 5**

- *Mức 1:* Bắt đầu với mọi bước đi sang phải (tổng `S_k`). Cần giảm đúng `S_k − T`.
- *Mức 2:* Giảm `2i` cho mỗi bước `i` đổi sang trái. Vậy cần tập bước có tổng `F = (S_k − T) / 2`.
- *Mức 3:* Duyệt `i` từ `k` xuống `1`: nếu `i ≤ F` thì đổi bước `i` sang trái, `F -= i`. Cuối cùng `F = 0`. Nếu `target < 0` đổi dấu cả dãy.

## 19. Đáp án tham khảo

### Đáp án Bài 1

- `reachNumber(2) = 3`.
- `reachNumber(-3) = 2` (đối xứng với `3`).
- `reachNumber(5) = 5` (`k0 = 3` lệch chẵn/lẻ, thêm 2 bước).

In [40]:
assert Solution().reachNumber(2) == 3
assert Solution().reachNumber(-3) == 2
assert Solution().reachNumber(5) == 5
print("Bài 1 OK")

Bài 1 OK


### Đáp án Bài 2

`target = 4`: `k = 1`: `S = 1 < 4`; `k = 2`: `S = 3 < 4`; `k = 3`: `S = 6 ≥ 4`, `S − 4 = 2` chẵn ✓. `F = 1`: đổi bước `1` sang trái: `−1 + 2 + 3 = 4`. Đáp án `k = 3`, bước đi sang trái: `[1]`.

In [41]:
hand_k = 3
hand_left_steps = [1]
assert hand_k == Solution().reachNumber(4)
assert hand_k * (hand_k + 1) // 2 - 2 * sum(hand_left_steps) == 4
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [42]:
def exercise_3_solution(target: int) -> int:
    target = abs(target)
    k = s = 0
    while s < target or (s - target) % 2:
        k += 1
        s += k
    return k

assert exercise_3_solution(2) == 3
assert exercise_3_solution(3) == 2
assert exercise_3_solution(-5) == 5
assert exercise_3_solution(10 ** 9) == 44723
for t in range(-200, 201):
    if t:
        assert exercise_3_solution(t) == reach_positions(t)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [43]:
def exercise_4_solution(target: int) -> int:
    T = abs(target)
    k = (isqrt(8 * T + 1) - 1) // 2
    if k * (k + 1) // 2 < T:
        k += 1
    while (k * (k + 1) // 2 - T) % 2:
        k += 1
    return k

assert exercise_4_solution(2) == 3
assert exercise_4_solution(3) == 2
assert exercise_4_solution(-10 ** 9) == 44723
assert exercise_4_solution(5) == 5
for t in range(-3000, 3001):
    if t:
        assert exercise_4_solution(t) == Solution().reachNumber(t)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [44]:
def exercise_5_solution(target: int) -> List[int]:
    T = abs(target)
    k = s = 0
    while s < T or (s - T) % 2:
        k += 1
        s += k
    F = (s - T) // 2
    signs = [1] * k
    for i in range(k, 0, -1):
        if i <= F:
            signs[i - 1] = -1
            F -= i
    assert F == 0
    return [-d for d in signs] if target < 0 else signs

for t in list(range(-500, 0)) + list(range(1, 501)) + [10 ** 9, -10 ** 9]:
    sg = exercise_5_solution(t)
    assert len(sg) == Solution().reachNumber(t)
    assert sum(d * i for i, d in enumerate(sg, 1)) == t
    assert set(sg) <= {1, -1}
print(exercise_5_solution(2), exercise_5_solution(-3))
print("Bài 5 OK")

[1, -1, 1] [-1, -1]
Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def reachNumber(self, target: int) -> int:
        target = abs(target)             # đối xứng
        k = s = 0                        # s = 1 + 2 + ... + k
        while s < target or (s - target) % 2:
            k += 1
            s += k
        return k
```

Câu thần chú: **"Lấy trị tuyệt đối; cộng 1, 2, 3, ... cho đến khi tổng ≥ đích và hiệu với đích là số chẵn."**

### Checklist trước khi submit

- [ ] Có `abs(target)` (đích âm đối xứng với đích dương).
- [ ] Điều kiện dừng gồm **cả hai**: `s >= target` **và** `(s - target)` chẵn.
- [ ] Không đoán số bước thêm: để vòng lặp tự kiểm tra (có thể phải thêm 2 bước).
- [ ] Hiểu vì sao đúng: đổi hướng bước `i` giảm `2i` (chẵn) + mọi số `0..S_k` là tổng con của `{1..k}`.
- [ ] Đã thử `target = ±1, ±2, ±3, 5` và `±10^9` (đáp án `44723`).
- [ ] Đã đối chiếu oracle tập vị trí trên `T` nhỏ.
- [ ] Skeleton đúng: `class Solution`, method `reachNumber(self, target: int) -> int`.